
# 02 - Data Cleaning
## Financial Transaction Analytics – Money Flows & Fraud Detection

This notebook focuses on validating, cleaning, and preparing the PaySim financial transactions dataset for exploratory data analysis.

### Objectives
- Load the raw dataset.
- Verify missing values and duplicate records.
- Check data types and value consistency.
- Investigate anomalous transactions and account balances.
- Document data quality issues and cleaning decisions.
- Export the validated dataset to the processed data directory.

### Scope
This notebook is limited to data quality assessment and preparation. Transaction patterns, monetary flows, fraud rates, and statistical distributions will be analyzed in `03_exploratory_data_analysis.ipynb`.


In [1]:
# Import libraries for data cleaning and validation
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Define the path to the raw dataset
file_path = Path("../data/raw/PS_20174392719_1491204439457_log.csv")

# Load the raw financial transactions dataset
df = pd.read_csv(file_path)

In [4]:
# Count transactions with negative amounts
(df["amount"] < 0).sum()

np.int64(0)

In [5]:
# Count transactions with zero amounts
(df["amount"] == 0).sum()

np.int64(16)

In [6]:
# Count negative values in account balance columns
balance_columns = [
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

(df[balance_columns] < 0).sum()

oldbalanceOrg     0
newbalanceOrig    0
oldbalanceDest    0
newbalanceDest    0
dtype: int64

In [7]:
# Check origin balance inconsistencies in TRANSFER transactions
transfers = df[df["type"] == "TRANSFER"]

inconsistent_transfers = (
    transfers["oldbalanceOrg"] - transfers["amount"]
    != transfers["newbalanceOrig"]
)

inconsistent_transfers.sum()

np.int64(517217)

In [8]:
# Count TRANSFER transactions with inconsistent origin balances
expected_balance = transfers["oldbalanceOrg"] - transfers["amount"]

inconsistent_transfers = ~np.isclose(
    expected_balance,
    transfers["newbalanceOrig"],
    rtol=0,
    atol=0.01
)

inconsistent_transfers.sum()

np.int64(508782)

In [9]:
# Inspect TRANSFER transactions with inconsistent origin balances
transfers.loc[
    inconsistent_transfers,
    ["amount", "oldbalanceOrg", "newbalanceOrig", "isFraud"]
].head(10)

,amount,oldbalanceOrg,newbalanceOrig,isFraud
19,215310.30,705.00,0.0,0
24,311685.89,10835.00,0.0,0
78,42712.39,10363.39,0.0,0
79,77957.68,0.00,0.0,0
80,17231.46,0.00,0.0,0
81,78766.03,0.00,0.0,0
82,224606.64,0.00,0.0,0
83,125872.53,0.00,0.0,0
84,379856.23,0.00,0.0,0
85,1505626.01,0.00,0.0,0


In [10]:
# Count inconsistent transfers with insufficient initial balance
(
    transfers.loc[inconsistent_transfers, "oldbalanceOrg"]
    < transfers.loc[inconsistent_transfers, "amount"]
).sum()

np.int64(507512)

In [11]:
# Inspect inconsistent transfers with sufficient initial balance
other_inconsistencies = (
    inconsistent_transfers
    & (transfers["oldbalanceOrg"] >= transfers["amount"])
)

transfers.loc[
    other_inconsistencies,
    ["amount", "oldbalanceOrg", "newbalanceOrig", "isFraud"]
].head(10)

,amount,oldbalanceOrg,newbalanceOrig,isFraud
1822,169106.97,575667.54,575667.54,0
9248,423234.24,1183874.19,760639.94,0
11247,33803.19,763336.90,729533.72,0
14904,248768.20,310245.27,61477.06,0
16887,9679.61,694583.77,684904.15,0
19223,202143.74,434299.82,232156.07,0
22242,533650.65,1728907.45,1195256.81,0
23117,425371.92,1449232.72,1023860.79,0
25516,1183182.40,1618668.70,435486.29,0
26751,181888.14,850699.86,668811.71,0


In [12]:
# Calculate absolute balance differences for the remaining transfers
balance_differences = (
    transfers.loc[other_inconsistencies, "oldbalanceOrg"]
    - transfers.loc[other_inconsistencies, "amount"]
    - transfers.loc[other_inconsistencies, "newbalanceOrig"]
).abs()

balance_differences.describe()

count    1.270000e+03
mean     6.138164e+04
std      6.671671e+05
min      1.000000e-02
25%      1.000000e-02
50%      1.000000e-02
75%      1.000000e-02
max      1.000000e+07
dtype: float64

In [13]:
# Count substantial balance inconsistencies
(balance_differences > 1).sum()

np.int64(17)

In [14]:
# Inspect transfers with substantial balance inconsistencies
transfers.loc[
    balance_differences[balance_differences > 1].index,
    ["amount", "oldbalanceOrg", "newbalanceOrig", "isFraud"]
]

,amount,oldbalanceOrg,newbalanceOrig,isFraud
1822,169106.97,575667.54,575667.54,0
2736446,4953893.08,4953893.08,4953893.08,1
3247297,1343002.08,1343002.08,1343002.08,1
3760288,536624.41,536624.41,536624.41,1
5563713,4892193.09,4892193.09,4892193.09,1
5996407,10000000.00,19585040.37,19585040.37,1
5996409,9585040.37,19585040.37,19585040.37,1
6168499,3576297.10,3576297.10,3576297.10,1
6205439,353874.22,353874.22,353874.22,1
6266413,2542664.27,2542664.27,2542664.27,1


In [15]:
# Check fraud flags for transfers with substantial balance inconsistencies
transfers.loc[
    balance_differences[balance_differences > 1].index,
    ["isFraud", "isFlaggedFraud"]
].value_counts()

isFraud  isFlaggedFraud
1        1                 16
0        0                  1
Name: count, dtype: int64

### Origin Balance Consistency – TRANSFER Transactions

The consistency check identified 508,782 TRANSFER transactions whose recorded origin balances differ from the expected balances by more than 0.01 monetary units.

Key findings:
- 507,512 transactions have insufficient initial balances.
- 1,253 additional transactions show differences of at most 1 monetary unit.
- 17 transactions show differences greater than 1 monetary unit.
- Of these 17 transactions, 16 are fraudulent and flagged by the simulator.

**Cleaning decision:** All transactions are retained. Balance inconsistencies are documented rather than automatically corrected, since they may reflect PaySim simulation behavior or fraud-related patterns.

In [16]:
# Check the unique values of fraud indicators
df[["isFraud", "isFlaggedFraud"]].nunique()


isFraud           2
isFlaggedFraud    2
dtype: int64

In [18]:
# Count invalid values in fraud indicators
(~df[["isFraud", "isFlaggedFraud"]].isin([0, 1])).sum()

isFraud           0
isFlaggedFraud    0
dtype: int64

In [19]:
# Count transactions with invalid transaction types
valid_types = ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"]

(~df["type"].isin(valid_types)).sum()

np.int64(0)

In [20]:
# Count transactions with invalid simulation steps
(df["step"] < 1).sum()

np.int64(0)

In [21]:
# Check the minimum and maximum simulation steps
df["step"].agg(["min", "max"])

min      1
max    743
Name: step, dtype: int64

## Data Cleaning Summary

The following data quality checks were completed:

- Missing values: none detected.
- Duplicate records: none detected.
- Negative transaction amounts: none.
- Zero-amount transactions: 16, retained.
- Negative account balances: none.
- Transaction types: all valid.
- Fraud indicators: binary values (0 and 1) only.
- Simulation steps: valid range from 1 to 743.
- TRANSFER origin balance inconsistencies: investigated and documented.

### Cleaning Decisions

No records were removed or modified.

All original transactions are retained to preserve the characteristics of the PaySim simulation, including potentially fraud-related anomalies.

The validated dataset will be exported without altering the original raw CSV.

In [22]:
# Define the output path for the validated dataset
output_path = Path("../data/processed/paysim_validated.csv")

# Export the validated dataset without the DataFrame index
df.to_csv(output_path, index=False)

In [23]:
# Check that the validated CSV file exists
output_path.is_file()

True